In [1]:
import math
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- FUNGSI INTI PERHITUNGAN ---
def hitung_hidrolika(L, D, Q_lps, H_pompa, C_hw, Elev_LWL, Elev_Outlet, K_minor, epsilon):
    # Konversi
    Q = Q_lps / 1000  # LPS ke m3/s
    g = 9.81
    nu = 1.002e-6
    
    # Kecepatan
    A = 0.25 * math.pi * (D**2)
    V = Q / A
    
    # Hazen-Williams
    hf_hw = 10.67 * (Q**1.852 / (C_hw**1.852 * D**4.871)) * L
    
    # Darcy-Weisbach
    Re = (V * D) / nu
    if Re > 2000:
        f = 0.25 / (math.log10((epsilon / (3.7 * D)) + (5.74 / (Re**0.9))))**2
    else:
        f = 64 / Re if Re > 0 else 0
    hf_dw = f * (L / D) * (V**2 / (2 * g))
    
    # Minor & Statis
    hm = K_minor * (V**2 / (2 * g))
    head_statis = Elev_Outlet - Elev_LWL
    
    # Residual (Menggunakan Darcy-Weisbach untuk akurasi)
    total_loss = hf_dw + hm
    residual = (Elev_LWL + H_pompa) - total_loss - Elev_Outlet
    
    return V, hf_hw, hf_dw, hm, head_statis, residual

# --- UI COMPONENTS (WIDGETS) ---
style = {'description_width': 'initial'}
L_w = widgets.FloatText(value=190.6, description='Panjang Pipa (m):', style=style)
D_w = widgets.FloatText(value=0.40, description='Diameter (m):', style=style)
Q_w = widgets.FloatText(value=250.0, description='Debit (LPS):', style=style)
H_p = widgets.FloatText(value=15.0, description='Head Pompa (m):', style=style)
C_w = widgets.IntSlider(value=150, min=50, max=150, description='Koef. H-W:', style=style)
LWL_w = widgets.FloatText(value=30.53, description='Elev. LWL (m):', style=style)
Out_w = widgets.FloatText(value=38.17, description='Elev. Outlet (m):', style=style)
K_w = widgets.FloatText(value=5.65, description='Total Koef Fitting:', style=style)

btn = widgets.Button(description="Hitung Sekarang", button_style='success')
out = widgets.Output()

def on_button_clicked(b):
    with out:
        clear_output()
        V, hf_hw, hf_dw, hm, h_statis, res = hitung_hidrolika(
            L_w.value, D_w.value, Q_w.value, H_p.value, C_w.value, LWL_w.value, Out_w.value, K_w.value, 0.000015
        )
        print(f"{'--- HASIL KALKULASI ---':^30}")
        print(f"Kecepatan: {V:.2f} m/s")
        print(f"Major Loss (DW): {hf_dw:.2f} m")
        print(f"Minor Loss: {hm:.2f} m")
        print(f"Head Statis: {h_statis:.2f} m")
        print("-" * 30)
        color = "\033[92m" if res > 0 else "\033[91m"
        print(f"HEAD RESIDUAL: {color}{res:.2f} m\033[0m")

btn.on_click(on_button_clicked)

# --- DISPLAY ---
display(widgets.VBox([
    widgets.HTML("<h2>Kalkulator Head Residual Online</h2>"),
    L_w, D_w, Q_w, H_p, C_w, LWL_w, Out_w, K_w, btn, out
]))